In [77]:
import pandas as pd
import os
import numpy as np
from datetime import datetime, timedelta


# Caminhos de entrada e saída
json_path = "noticias.json"
output_dir = "silver-json-to_parquet/noticias_parquet"

# Criar pasta de saída se não existir
os.makedirs(output_dir, exist_ok=True)
print("Lendo Json")

df = pd.read_json(json_path)
n = len(df)
hoje = datetime.today()
um_ano_atras = hoje - timedelta(days=365)

df["data_processamento"] = pd.to_datetime(
    np.random.uniform(um_ano_atras.timestamp(), hoje.timestamp(), n),
    unit="s"
)
df["data_processamento"] = df["data_processamento"].dt.normalize()
df["data_processamento"] = df["data_processamento"].dt.date
df['ano'] = pd.to_datetime(df["data_processamento"]).dt.year
df["ano"] = df["ano"].astype(int)
df["mes"] = pd.to_datetime(df["data_processamento"]).dt.month
df["mes"] = df["mes"].astype(int)
df = df.fillna('')
df["data_publicacao"] = df["data_publicacao"].replace('—','')

total_parquet = 0
anos_processados = []

print("Gerando arquivos Parquet por ano:")
for ano, grupo in df.groupby('ano'):
    for mes, grupo in df.groupby('mes'):
        path_ano = os.path.join(output_dir, f"ano={ano}", f"mes={mes}" )
        os.makedirs(path_ano, exist_ok=True)

        parquet_path = os.path.join(path_ano, f"dados_noticias_{ano}_{mes}.parquet")
        grupo = grupo.astype({
            "ano": "int64",
            "mes": "int64",
            "titulo": "string",
            "link": "string",
            "descricao": "string",
            "data_publicacao": "string",
        })
        grupo.to_parquet(
                parquet_path, engine='pyarrow', index=False,
                use_dictionary={"ano": False, "mes": False},
                coerce_timestamps="ms",
                allow_truncated_timestamps=True,
            )

        qtd = len(grupo)
        total_parquet += qtd
        anos_processados.append((ano, qtd))
        print(f" Ano/Mesref: {ano}/{mes}: {qtd:,} registros → {parquet_path}")



Lendo Json
Gerando arquivos Parquet por ano:
 Ano/Mesref: 2024/1: 25 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=1/dados_noticias_2024_1.parquet
 Ano/Mesref: 2024/2: 17 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=2/dados_noticias_2024_2.parquet
 Ano/Mesref: 2024/3: 24 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=3/dados_noticias_2024_3.parquet
 Ano/Mesref: 2024/4: 20 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=4/dados_noticias_2024_4.parquet
 Ano/Mesref: 2024/5: 14 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=5/dados_noticias_2024_5.parquet
 Ano/Mesref: 2024/6: 13 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=6/dados_noticias_2024_6.parquet
 Ano/Mesref: 2024/7: 14 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=7/dados_noticias_2024_7.parquet
 Ano/Mesref: 2024/8: 20 registros → silver-json-to_parquet/noticias_parquet/ano=2024/mes=8/dados_noticias_20

<H2> Validação de conteudo

In [78]:
inicio_ano = 2020
fim = 2025
df_parquet = pd.DataFrame()

while inicio_ano <= fim:
    ano = str(inicio_ano)
    print(f"Lendo ano de {ano}")
    mes = 1
    fim_mes = 12
    while mes <= fim_mes:
        
        print(f"Lendo {mes} mes")
        try:
            df_parquet_temp = pd.read_parquet(f"silver-json-to_parquet/noticias_parquet/ano={ano}/mes={mes}/dados_noticias_{ano}_{mes}.parquet")
            df_parquet = pd.concat([df_parquet, df_parquet_temp], ignore_index=True)
            print(f"Leitura {ano}/{mes} concluida")
            mes = mes + 1
        except Exception as e:
            # trata o erro
            print("Ocorreu um erro:", e)
        finally:
            mes = mes + 1
    inicio_ano = inicio_ano + 1

Lendo ano de 2020
Lendo 1 mes
Ocorreu um erro: [Errno 2] No such file or directory: 'silver-json-to_parquet/noticias_parquet/ano=2020/mes=1/dados_noticias_2020_1.parquet'
Lendo 2 mes
Ocorreu um erro: [Errno 2] No such file or directory: 'silver-json-to_parquet/noticias_parquet/ano=2020/mes=2/dados_noticias_2020_2.parquet'
Lendo 3 mes
Ocorreu um erro: [Errno 2] No such file or directory: 'silver-json-to_parquet/noticias_parquet/ano=2020/mes=3/dados_noticias_2020_3.parquet'
Lendo 4 mes
Ocorreu um erro: [Errno 2] No such file or directory: 'silver-json-to_parquet/noticias_parquet/ano=2020/mes=4/dados_noticias_2020_4.parquet'
Lendo 5 mes
Ocorreu um erro: [Errno 2] No such file or directory: 'silver-json-to_parquet/noticias_parquet/ano=2020/mes=5/dados_noticias_2020_5.parquet'
Lendo 6 mes
Ocorreu um erro: [Errno 2] No such file or directory: 'silver-json-to_parquet/noticias_parquet/ano=2020/mes=6/dados_noticias_2020_6.parquet'
Lendo 7 mes
Ocorreu um erro: [Errno 2] No such file or directory

In [79]:
df_parquet

,titulo,link,descricao,data_publicacao,data_processamento,ano,mes
0,Inadimplência atinge em setembro maior patamar...,https://www.cnnbrasil.com.br/economia/macroeco...,8 de out. de 2025 — A proporção de famílias co...,8 de out. de 2025—,2025-01-07,2025,1
1,Endividamento cresce no Brasil e atinge 77% da...,https://www.youtube.com/watch?v=GGqq6uhCXVQ,"O endividamento cresceu no Brasil , mas a inad...",,2025-01-12,2025,1
2,Parecer rico é nova moda dos endividados - E-I...,https://einvestidor.estadao.com.br/colunas/fab...,há 3 dias — O Brasil vive a era do endividamen...,,2025-01-04,2025,1
3,Famílias brasileiras reduzem inadimplência e ...,https://www.terra.com.br/economia/familias-bra...,22 de jan. de 2025 — A proporção de famílias c...,22 de jan. de 2025—,2025-01-24,2025,1
4,Recorde: 76% das famílias brasileiras estão en...,https://www.cartacapital.com.br/economia/recor...,18 de jan. de 2022 — Cartão de crédito represe...,18 de jan. de 2022—,2025-01-05,2025,1
...,...,...,...,...,...,...,...
215,Demanda por voos internacionais aumenta no Brasil,https://istoe.com.br/demanda-por-voos-internac...,3 de mar. de 2018 — Nas companhias brasileiras...,3 de mar. de 2018—,2024-11-04,2024,11
216,Número de voos internacionais para o Brasil em...,https://blog.123milhas.com/numero-de-voos-inte...,11 de ago. de 2022 — Número de voos internacio...,11 de ago. de 2022—,2024-11-06,2024,11
217,Voos internacionais consolidam o Pará como pon...,https://agenciapara.com.br/noticia/71929/voos-...,há 1 dia — A partir do próximo dia 27 de outub...,,2024-11-08,2024,11
218,Senado se antecipa à Câmara e aprova bagagem d...,https://exame.com/brasil/senado-se-antecipa-a-...,há 4 dias — — Nós aprovamos um projeto que imp...,,2024-11-04,2024,11


In [83]:
df_vazios.eq('').sum()

titulo                  0
link                    0
descricao              16
data_publicacao       118
data_processamento      0
ano                     0
mes                     0
dtype: int64